# 02｜PSF/OTF、像差与共聚焦针孔测试（中文）

本教程演示针孔大小、像差参数和 pinhole_mode 对结果的影响。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyotf.microscope import ConfocalMicroscope

## 第 1 步：构建两个针孔大小不同的共聚焦模型

In [ ]:
base_kwargs = dict(model="hanser", na=1.1, ni=1.33, wl=520, size=64, pixel_size=80, oversample_factor=1)
confocal_open = ConfocalMicroscope(wl_exc=488, pinhole_size=2.0, **base_kwargs)
confocal_small = ConfocalMicroscope(wl_exc=488, pinhole_size=0.4, **base_kwargs)
print(confocal_open.PSF.shape, confocal_small.PSF.shape)

## 第 2 步：比较轴向积分曲线

In [ ]:
prof_open = confocal_open.PSF.sum(axis=(1,2))
prof_small = confocal_small.PSF.sum(axis=(1,2))
plt.figure(figsize=(6,4))
plt.plot(prof_open, label="2.0 AU")
plt.plot(prof_small, label="0.4 AU")
plt.title("不同针孔下的轴向积分曲线")
plt.xlabel("z index")
plt.ylabel("integrated intensity")
plt.legend()
plt.show()

## 第 3 步：加入分路径像差（激发/检测）

In [ ]:
confocal_ab = ConfocalMicroscope(
    wl_exc=488, pinhole_size=1.0,
    aberrations_em={"defocus": 0.2},
    aberrations_exc={"vertical astigmatism": 0.1},
    **base_kwargs
)
z0 = confocal_ab.PSF.shape[0] // 2
plt.figure(figsize=(5,4))
plt.imshow(confocal_ab.PSF[z0], cmap="magma")
plt.title("加入像差后的 confocal 中心切片")
plt.colorbar()
plt.show()

## 第 4 步：比较 pinhole_mode

In [ ]:
confocal_obj = ConfocalMicroscope(wl_exc=488, pinhole_size=1.2, pinhole_mode="object", **base_kwargs)
confocal_det = ConfocalMicroscope(wl_exc=488, pinhole_size=1.2, pinhole_mode="detector", **base_kwargs)
print("object sum:", confocal_obj.PSF.sum())
print("detector sum:", confocal_det.PSF.sum())